# Low Chin Tone Detection from EMG — **Version 4**
**What’s new (fixes your miss-detections):**
- **Sliding RMS (0.2s & 1.0s) with 0.1s hop** → no more “window-boundary” blind spots
- **Adaptive threshold (rolling median + MAD)** → robust to baseline drift & amplitude variability
- **Hysteresis (start/end thresholds) + gap-merge** → stable event on/off, fewer flickers
- **Two-scale logic** (short & long envelopes) → catches **short, tall spikes** *and* **longer, thicker bursts**
- **Event–segment mapping** → 3s summaries mark a segment “HIGH” if any event overlaps it

> The notebook keeps your EDF I/O & XML conventions while upgrading detection. If `sleep_stage` is available in your environment, it will load EDF exactly like your V3. Otherwise, you can swap in your own loader.

In [ ]:

import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt
from datetime import timedelta, datetime
import xml.etree.ElementTree as ET

# Optional: use your existing sleep_stage loader (as in your V3)
try:
    sys.path.append('/home/honeynaps/data/shared/integrate/sleep_stage')
    from sleep_stage.modules.iofiles import edf as edf_io
    HAVE_SLEEP_STAGE = True
except Exception as e:
    HAVE_SLEEP_STAGE = False
    print("[WARN] sleep_stage not available in this environment. "
          "Replace the EDF loader with your own if needed.")

In [ ]:

def butter_highpass(cutoff_hz, fs, order=4):
    nyq = 0.5 * fs
    norm = cutoff_hz / nyq
    b, a = butter(order, norm, btype='highpass')
    return b, a

def highpass_filter(x, fs, cutoff_hz=5.0, order=4):
    if cutoff_hz is None or cutoff_hz <= 0:
        return x
    b, a = butter_highpass(cutoff_hz, fs, order=order)
    # zero-phase
    return filtfilt(b, a, x)

def rms_envelope(signal, fs, win_s=0.2, hop_s=0.1):
    """
    RMS envelope via moving-average of squared signal.
    Returns (times, envelope).
    """
    win = max(1, int(round(win_s * fs)))
    hop = max(1, int(round(hop_s * fs)))
    if win <= 1:
        # fallback to absolute value if extremely small window
        env = np.abs(signal)
        t = np.arange(len(env)) / fs
        return t[::hop], env[::hop]
    kernel = np.ones(win, dtype=float) / float(win)
    sq = signal.astype(float)**2
    # same-mode moving avg
    movavg = np.convolve(sq, kernel, mode='same')
    env = np.sqrt(np.maximum(movavg, 0.0))
    t = np.arange(len(env)) / fs
    return t[::hop], env[::hop]

def rolling_median_mad(x, win):
    """
    Rolling (centered) median and MAD (scaled) for adaptive baseline.
    Returns (median, mad_scaled). Edges are padded with nearest valid.
    """
    s = pd.Series(x)
    med = s.rolling(win, center=True, min_periods=1).median()
    mad = (s - med).abs().rolling(win, center=True, min_periods=1).median()
    mad_scaled = 1.4826 * mad  # ~N(0,1) consistency
    med = med.fillna(method='bfill').fillna(method='ffill')
    mad_scaled = mad_scaled.fillna(method='bfill').fillna(method='ffill')
    # Avoid zero MAD (gives infinite z).
    mad_scaled = mad_scaled.replace(0, np.median(mad_scaled[mad_scaled>0]) if (mad_scaled>0).any() else 1e-8)
    return med.values, mad_scaled.values

In [ ]:

from dataclasses import dataclass

@dataclass
class DetectConfig:
    fs: int = 50
    hp_cut_hz: float = 5.0         # high-pass to remove drift (safe with fs=50)
    short_win_s: float = 0.2       # 200 ms
    long_win_s: float = 1.0        # 1 s
    hop_s: float = 0.1             # 100 ms frame
    baseline_win_s: float = 60.0   # rolling baseline window (60 s)
    # Hysteresis thresholds (robust z-score)
    z_start_short: float = 4.0
    z_end_short: float = 2.5
    z_start_long: float = 3.5
    z_end_long: float = 2.0
    # Ratio thresholds (envelope / baseline)
    r_start_short: float = 2.5
    r_end_short: float = 1.6
    r_start_long: float = 2.0
    r_end_long: float = 1.4
    # Event constraints
    min_event_dur_s: float = 0.10
    merge_gap_s: float = 0.15

def detect_emg_events(raw_emg, meas_date, cfg: DetectConfig):
    fs = cfg.fs
    x = raw_emg.astype(float)
    # DC removal + high-pass
    x = x - np.median(x)
    if cfg.hp_cut_hz and cfg.hp_cut_hz > 0:
        x = highpass_filter(x, fs, cutoff_hz=cfg.hp_cut_hz, order=4)

    # Envelopes (two scales)
    t_s, env_s = rms_envelope(x, fs, win_s=cfg.short_win_s, hop_s=cfg.hop_s)
    t_l, env_l = rms_envelope(x, fs, win_s=cfg.long_win_s,  hop_s=cfg.hop_s)

    # Rolling baselines (median + MAD) in frames
    base_win_frames = max(3, int(round(cfg.baseline_win_s / cfg.hop_s)))

    b_s, mad_s = rolling_median_mad(env_s, base_win_frames)
    b_l, mad_l = rolling_median_mad(env_l, base_win_frames)

    eps = 1e-8
    z_s = (env_s - b_s) / (mad_s + eps)
    z_l = (env_l - b_l) / (mad_l + eps)

    r_s = env_s / np.maximum(b_s, eps)
    r_l = env_l / np.maximum(b_l, eps)

    # Hysteresis + union of two scales
    start_mask = (z_s > cfg.z_start_short) | (z_l > cfg.z_start_long) |                  (r_s > cfg.r_start_short) | (r_l > cfg.r_start_long)
    end_mask   = (z_s < cfg.z_end_short) & (z_l < cfg.z_end_long) &                  (r_s < cfg.r_end_short) & (r_l < cfg.r_end_long)

    # Iterate frames to form events
    in_evt = False
    evt_list = []
    evt_start = None
    for i in range(len(t_s)):
        if not in_evt and start_mask[i]:
            in_evt = True
            evt_start = t_s[i]
        elif in_evt and end_mask[i]:
            in_evt = False
            evt_end = t_s[i]
            if evt_end - evt_start >= cfg.min_event_dur_s:
                evt_list.append((evt_start, evt_end))

    # If ended while still in event
    if in_evt:
        evt_end = t_s[-1]
        if evt_end - evt_start >= cfg.min_event_dur_s:
            evt_list.append((evt_start, evt_end))

    # Merge nearby events
    merged = []
    if evt_list:
        cur_s, cur_e = evt_list[0]
        for s, e in evt_list[1:]:
            if s - cur_e <= cfg.merge_gap_s:
                cur_e = max(cur_e, e)
            else:
                merged.append((cur_s, cur_e))
                cur_s, cur_e = s, e
        merged.append((cur_s, cur_e))

    # Compute average RMS (short envelope) for each event
    # Map event window to indices in env_s (frame resolution)
    def frame_at(t): return int(round(t / cfg.hop_s))

    events = []
    for s, e in merged:
        i0 = frame_at(max(0.0, s))
        i1 = frame_at(min(t_s[-1], e))
        i1 = max(i1, i0 + 1)
        avg_rms = float(np.mean(env_s[i0:i1]))
        events.append({
            "start": float(s),
            "end": float(e),
            "duration": float(e - s),
            "avg_rms": avg_rms,
            "type": "HIGH"  # high-chin tone event
        })

    debug = {
        "t_short": t_s, "env_short": env_s, "base_short": b_s, "z_short": z_s, "r_short": r_s,
        "t_long":  t_l, "env_long":  env_l, "base_long":  b_l, "z_long":  z_l, "r_long":  r_l,
        "events": events
    }
    return events, debug

In [ ]:

def load_edf_raw_v4(edf_path, fs=50):
    """
    Loads EDF via your sleep_stage io module, returns:
      - raw CHIN as a 1D numpy array (float)
      - meas_date (datetime, naive)
      - channel names list
    """
    if not HAVE_SLEEP_STAGE:
        raise RuntimeError("sleep_stage edf_io not available here. "
                           "Please run this notebook in your environment where it is installed.")
    edf, n_missing_ch = edf_io.load(
        path       = edf_path, 
        preload    = True, 
        resample   = fs, 
        preset     = "STAGENET", 
        exclude    = True,
        missing_ch = 'raise'
    )
    meas_date = edf.info['meas_date'].replace(tzinfo=None)
    data = edf.get_data()
    ch_names = edf.ch_names

    SID_MAP = { 
        'F3-':'F3_2', 'F4-':'F4_1', 'C3-':'C3_2', 'C4-':'C4_1', 
        'O1-':'O1_2', 'O2-':'O2_1', 'LOC':'LOC', 'ROC':'ROC', 'EMG':'CHIN'
    }
    sigs = {}
    for i, nm in enumerate(ch_names):
        if nm in SID_MAP:
            sigs[SID_MAP[nm]] = data[i]
        else:
            # fallback map by prefix (as in your V3)
            sigs[SID_MAP.get(nm[:3], nm[:3])] = data[i]
    if 'CHIN' not in sigs:
        raise RuntimeError("EMG channel 'CHIN' not found. Please adjust mapping.")

    chin = sigs['CHIN'].astype(float)
    # Convert to microvolts if needed (same scaling as your V3)
    chin = chin * 1000.0
    return chin, meas_date, ch_names

In [ ]:

def map_events_to_segments(emg_len, fs, seg_sec, events):
    """
    For 3-second segments, mark segment 'HIGH' if any event overlaps it.
    Returns list of dicts: {'start':t0,'duration':seg_sec,'type':'HIGH'/'LOW','avg_rms':...}
    """
    n_seg = int(emg_len // (seg_sec * fs))
    out = []
    for i in range(n_seg):
        s = i * seg_sec
        e = s + seg_sec
        # Overlap test
        overlap = False
        # find intersect with any event
        for ev in events:
            if (ev['start'] < e) and (ev['end'] > s):
                overlap = True
                break
        out.append({
            "start": s,
            "duration": seg_sec,
            "type": "HIGH" if overlap else "LOW",
            "avg_rms": None  # will fill with segment RMS if provided
        })
    return out

def segment_rms(x, fs, seg_sec=3.0):
    L = len(x)
    seg_len = int(round(seg_sec * fs))
    n_seg = L // seg_len
    val = []
    for i in range(n_seg):
        seg = x[i*seg_len:(i+1)*seg_len]
        if len(seg) == seg_len:
            val.append(float(np.sqrt(np.mean(seg.astype(float)**2))))
    return np.array(val, dtype=float)

def save_events_xml(meas_date, events, xml_path):
    root = ET.Element("annotationlist")
    if events:
        recording_dur = events[-1]['end']
    else:
        recording_dur = 0.0
    ET.SubElement(root, "recording_duration").text = f"{recording_dur:.6f}"

    for ev in events:
        onset_time = meas_date + timedelta(seconds=ev['start'])
        desc = f"{ev['type']}_CHIN_TONE_{ev['avg_rms']:.4f}"

        ann = ET.SubElement(root, "annotation")
        on_elem = ET.SubElement(ann, "onset")
        on_elem.text = onset_time.strftime("%Y-%m-%dT%H:%M:%S.%f")

        du_elem = ET.SubElement(ann, "duration")
        du_elem.text = f"{ev['duration']:.6f}"

        de_elem = ET.SubElement(ann, "description")
        de_elem.text = desc

        lo_elem = ET.SubElement(ann, "location")
        lo_elem.text = "EEG-EMG"

    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ", level=0)
    with open(xml_path, "wb") as fp:
        fp.write(b'<?xml version="1.0" encoding="UTF-8"?>\n')
        tree.write(fp, encoding="UTF-8", xml_declaration=False)
    return len(events)

In [ ]:

# ========== CONFIG ==========
fs = 50
SEG_SEC = 3.0                 # for summaries and per-3s labels
OUTPUT_DIR = "./output_v4"    # change as needed
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Two-scale adaptive detector
cfg = DetectConfig(
    fs=fs,
    hp_cut_hz=5.0,
    short_win_s=0.2, long_win_s=1.0, hop_s=0.1,
    baseline_win_s=60.0,
    z_start_short=4.0, z_end_short=2.5,
    z_start_long=3.5, z_end_long=2.0,
    r_start_short=2.5, r_end_short=1.6,
    r_start_long=2.0, r_end_long=1.4,
    min_event_dur_s=0.10,
    merge_gap_s=0.15
)

# ===== INPUT PATHS (edit to your env) =====
EDF_DIR = '/home/honeynaps/data/GOLDEN/EDF2'
MAX_FILES = 5  # for a quick run; set None to process all
edf_files = [f for f in os.listdir(EDF_DIR) if f.lower().endswith('.edf')]
edf_files.sort()
if MAX_FILES is not None:
    edf_files = edf_files[:MAX_FILES]

print(f"Found {len(edf_files)} EDF files to process")

results = []
for idx, fname in enumerate(edf_files, 1):
    edf_path = os.path.join(EDF_DIR, fname)
    print(f"\n[{idx}/{len(edf_files)}] Processing {fname}")

    # Load CHIN channel & meas_date
    chin, meas_date, ch_names = load_edf_raw_v4(edf_path, fs=fs)
    print(f"Loaded CHIN len={len(chin)} @ {fs} Hz | meas_date={meas_date}")

    # Detect events (HIGHs)
    events, dbg = detect_emg_events(chin, meas_date, cfg)

    # Fill avg_rms for 3s segments too, and mark HIGH/LOW by overlap
    seg_labels = map_events_to_segments(len(chin), fs, SEG_SEC, events)
    seg_rms = segment_rms(chin, fs, seg_sec=SEG_SEC)
    for i, seg in enumerate(seg_labels):
        if i < len(seg_rms):
            seg['avg_rms'] = float(seg_rms[i])

    # Save event-level XML
    base = os.path.splitext(fname)[0]
    xml_path = os.path.join(OUTPUT_DIR, f"{base}_CHIN_EVENTS_V4.xml")
    n_saved = save_events_xml(meas_date, events, xml_path)
    print(f"  Saved {n_saved} HIGH events -> {xml_path}")

    # Summary
    n_high_seg = sum(1 for s in seg_labels if s['type']=="HIGH")
    n_low_seg  = sum(1 for s in seg_labels if s['type']=="LOW")
    results.append({
        "file": fname, "n_events": len(events),
        "n_seg_high": n_high_seg, "n_seg_low": n_low_seg,
        "xml": xml_path
    })

print("\n=== SUMMARY ===")
for r in results:
    print(f"{r['file']}: events={r['n_events']} | seg(H/L)={r['n_seg_high']}/{r['n_seg_low']} -> {r['xml']}")
print("Done.")

In [ ]:

# OPTIONAL: visualize detection on a short interval for QA
# Pick the last processed file data still in memory: 'chin' and 'dbg' from above.
try:
    start_s, dur_s = 30.0, 30.0   # change view window here
    fs = cfg.fs
    t = np.arange(len(chin))/fs
    m0 = int(round(start_s * fs))
    m1 = int(round((start_s + dur_s) * fs))

    plt.figure(figsize=(14, 6))
    plt.plot(t[m0:m1], chin[m0:m1], linewidth=0.8)
    plt.title("Raw CHIN EMG")
    plt.xlabel("Time (s)"); plt.ylabel("µV")
    plt.show()

    # Envelopes & baseline on the same view (frame-rate downsampled)
    t_s = dbg['t_short']; env_s = dbg['env_short']; base_s = dbg['base_short']
    t_l = dbg['t_long'];  env_l = dbg['env_long'];  base_l = dbg['base_long']

    mask_s = (t_s >= start_s) & (t_s <= start_s + dur_s)
    mask_l = (t_l >= start_s) & (t_l <= start_s + dur_s)

    plt.figure(figsize=(14, 6))
    plt.plot(t_s[mask_s], env_s[mask_s], label="env_short")
    plt.plot(t_s[mask_s], base_s[mask_s], label="base_short", linestyle="--")
    plt.plot(t_l[mask_l], env_l[mask_l], label="env_long")
    plt.plot(t_l[mask_l], base_l[mask_l], label="base_long", linestyle="--")
    # Shade events
    for ev in dbg['events']:
        if ev['end'] < start_s or ev['start'] > start_s + dur_s:
            continue
        xs = [ev['start'], ev['end']]
        plt.axvspan(xs[0], xs[1], alpha=0.2)
    plt.legend()
    plt.title("RMS Envelopes (short/long) with baselines & detected events")
    plt.xlabel("Time (s)"); plt.ylabel("RMS (µV)")
    plt.show()
except Exception as e:
    print("Visualization skipped:", e)